In [1]:
# Import Libraries and Load the Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [2]:
# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [3]:
# Load the pre-trained model with ReLU activation
model = load_model('simple_rnn_imdb.h5')
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [4]:
model.get_weights()

[array([[-0.02737939, -0.01092201, -0.08998166, ...,  0.07232057,
         -0.04291259, -0.00380939],
        [-0.03832039,  0.01402455, -0.01497747, ..., -0.03726425,
         -0.03823216,  0.02890516],
        [-0.06198749,  0.0187113 , -0.03415491, ...,  0.02112179,
         -0.03409879, -0.00026017],
        ...,
        [ 0.04492278, -0.01900274,  0.01401649, ..., -0.05752571,
          0.06296973,  0.15864603],
        [ 0.13007765,  0.01785032,  0.1381564 , ..., -0.07663428,
          0.0890229 , -0.00633383],
        [-0.24553315, -0.07029694, -0.22797303, ...,  0.17699136,
         -0.20316647, -0.04597664]], shape=(10000, 128), dtype=float32),
 array([[ 0.08566102,  0.11628067, -0.13658632, ...,  0.07944065,
          0.15025367,  0.1092436 ],
        [-0.03020739, -0.03028656,  0.02465269, ..., -0.07570949,
          0.08139896, -0.0776209 ],
        [-0.10516849,  0.0305552 , -0.02388608, ...,  0.05302072,
         -0.07039174,  0.05264416],
        ...,
        [ 0.0911471

In [ ]:
# Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    import re
    # Clean punctuation and convert to lowercase
    cleaned_text = re.sub(r'[^\w\s]', '', text.lower())
    words = cleaned_text.split()
    
    # Prepend the start token (1) as the model expects
    encoded_review = [1]
    for word in words:
        idx = word_index.get(word)
        # Cap index to 10000 (max_features) or set to UNK (2) if not found
        if idx is None or idx + 3 >= 10000:
            encoded_review.append(2)
        else:
            encoded_review.append(idx + 3)
            
    padded_review = sequence.pad_sequences([encoded_review], maxlen=200)
    return padded_review


In [6]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]

In [7]:
# User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.9915096163749695
